# 🏅 AIMO3 Gold Medal Pipeline
## AI Mathematical Olympiad - Progress Prize 3

このノートブックは、AIMO3コンペティションで金メダルを狙うための完全なMLパイプラインです。

### 🔑 主要戦略 (AIMO2上位解法 + 最新研究を統合)

| コンポーネント | 手法 |
|---|---|
| **モデル** | Qwen3-32B / DeepSeek-R1-Distill-Qwen-32B (H100 x2) |
| **推論サーバー** | vLLM (高スループット並列推論) |
| **コア手法** | Tool-Integrated Reasoning (TIR) + Python実行 |
| **アンサンブル** | Self-Consistency (多数決) + 複数モデルアンサンブル |
| **後処理** | 堅牢なanswer extraction + 5桁整数への変換 |

### 📚 参考文献
- [Project Numina (AIMO1 1位)](https://github.com/project-numina/aimo-progress-prize)
- [NemoSkills (AIMO2 1位)](https://www.kaggle.com/competitions/ai-mathematical-olympiad-progress-prize-2/discussion/572760)
- [MuMath-Code Paper](https://arxiv.org/abs/2405.04214)
- [ToRA Paper (Tool-integrated Reasoning)](https://arxiv.org/abs/2309.17452)
- [DeepSeek-R1 Technical Report](https://arxiv.org/abs/2501.12948)

## 📦 Step 1: 環境セットアップ & パッケージインストール

In [ ]:
import subprocess
import sys
import os

# ==========================================
# GPU環境の確認
# ==========================================
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

# H100 x2 を想定 (Kaggle AIMO3 環境)
# AIMO3では最大2枚のH100 GPUが利用可能
NUM_GPUS = int(os.environ.get('NUM_GPUS', 2))
print(f'\n✅ 使用GPU数: {NUM_GPUS}')

In [ ]:
# ==========================================
# vLLM インストール (最新版)
# AIMO3 H100環境では vLLM >= 0.8.4 を推奨
# ==========================================
!pip install -q vllm>=0.8.4 
!pip install -q transformers>=4.47.0 accelerate datasets
!pip install -q sympy scipy numpy pandas
!pip install -q timeout-decorator

print('✅ インストール完了')

In [ ]:
# ==========================================
# ライブラリのインポート
# ==========================================
import os
import re
import sys
import ast
import math
import json
import time
import signal
import textwrap
import traceback
import contextlib
import io
from io import StringIO
from typing import Optional
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FuturesTimeoutError
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import sympy
from sympy import *

print('✅ ライブラリのインポート完了')

## ⚙️ Step 2: 設定クラス (Config)

すべてのハイパーパラメータをここで管理します。

In [ ]:
@dataclass
class AIMOConfig:
    """
    AIMO3パイプラインの全設定
    
    モデル選択の優先順位 (H100 x2 前提):
    1. Qwen3-32B              → 最も強力な数学推論、TIRとの相性◎
    2. DeepSeek-R1-Distill-Qwen-32B → AIMO2上位陣の定番
    3. Qwen2.5-Math-72B-Instruct → 数学特化72Bモデル
    """

    # --- モデル設定 ---
    # primary: メインの推論に使う最強モデル
    primary_model: str = "Qwen/Qwen3-32B"
    # secondary: アンサンブル用の2番手モデル
    secondary_model: str = "deepseek-ai/DeepSeek-R1-Distill-Qwen-32B"
    
    # --- vLLM設定 ---
    tensor_parallel_size: int = 2          # H100 x2 でテンソル並列
    max_model_len: int = 32768             # 最大コンテキスト長
    gpu_memory_utilization: float = 0.92   # GPUメモリ使用率
    dtype: str = "bfloat16"               # H100はbfloat16が最適
    enable_prefix_caching: bool = True     # プレフィックスキャッシュで高速化

    # --- サンプリング設定 (DeepSeek推奨設定を参考) ---
    # AIMO2 1位: temperature=0.6, N=32, depth=3
    # AIMO2 2位: temperature=0.7, N=64, depth=2  
    # 我々の設定: temperature=0.6, N=48, depth=4
    temperature: float = 0.6              # 0.5-0.7が推奨範囲
    top_p: float = 0.95
    max_tokens: int = 8192                # 1回の生成最大トークン
    
    # --- TIR (Tool-Integrated Reasoning) 設定 ---
    # N = 候補数 (多数決のサンプル数)
    # depth = コード実行のループ回数 (自己修正の深さ)
    num_candidates: int = 48              # 多数決の候補数 (Numina: 48)
    max_tir_depth: int = 4                # TIRの最大深さ (Numina: 4)
    code_timeout: int = 10               # コード実行タイムアウト (秒)
    
    # --- アンサンブル設定 ---
    use_ensemble: bool = True             # 複数モデルのアンサンブル
    primary_weight: float = 0.6           # primaryモデルの投票重み
    secondary_weight: float = 0.4         # secondaryモデルの投票重み
    
    # --- 答え形式 (AIMO3固有) ---
    # AIMO3は5桁の整数 (00000-99999)
    answer_mod: int = 100000              # 答えを100000で割った余り
    
    # --- その他 ---
    is_submission: bool = True            # Kaggle提出モードかどうか
    seed: int = 42
    debug: bool = False                   # デバッグモード


config = AIMOConfig()
print('✅ Config loaded:')
print(f'  Primary model: {config.primary_model}')
print(f'  Candidates (N): {config.num_candidates}')
print(f'  TIR depth (M): {config.max_tir_depth}')
print(f'  Tensor parallel: {config.tensor_parallel_size}')

## 🗂️ Step 3: データ読み込み

In [ ]:
import pandas as pd

# ==========================================
# Kaggle AIMO3 API を使ったデータ読み込み
# ==========================================
if config.is_submission:
    # 提出時: Kaggle APIから問題を取得
    import aimo  # Kaggle AIMO3 公式評価API
    problems_df = aimo.load_problems()
else:
    # ローカル開発時: サンプルデータを使用
    # 公式バリデーションセット (AIME問題など) でテスト
    from datasets import load_dataset
    
    # AIME 2024 validation set (難易度が近い)
    ds = load_dataset('AI-MO/aimo-validation-aime', split='train')
    problems_df = pd.DataFrame({
        'id': range(len(ds)),
        'problem': ds['problem'],
        'answer': ds['answer']  # ローカル評価用
    })

print(f'✅ 問題数: {len(problems_df)}')
print(f'\nサンプル問題:')
print(problems_df.iloc[0]['problem'][:200] + '...')

## 🐍 Step 4: Tool-Integrated Reasoning (TIR) エンジン

TIRは数学オリンピック問題解法の核心です。モデルがPythonコードを書き、実行し、結果を見て推論を続けます。

In [ ]:
# ==========================================
# 安全なコード実行環境
# ==========================================

class SafeCodeExecutor:
    """
    数学コードを安全に実行するサンドボックス。
    
    - タイムアウト制御
    - 標準出力キャプチャ
    - エラーハンドリング
    - sympy/numpy/scipy を利用可能
    """
    
    # 許可するモジュール (危険なモジュールを除外)
    SAFE_MODULES = {
        'math', 'cmath', 'decimal', 'fractions', 'statistics',
        'itertools', 'functools', 'operator', 'collections',
        'sympy', 'numpy', 'scipy', 'random', 'string',
        're', 'json', 'copy', 'heapq', 'bisect',
    }
    
    def __init__(self, timeout: int = 10):
        self.timeout = timeout
        # 実行環境の初期化 (sympy等をデフォルトで利用可能)
        self.global_env = {
            '__builtins__': {
                k: v for k, v in __builtins__.__dict__.items()
                if k not in {'exec', 'eval', 'compile', 'open', 
                             '__import__', 'input', 'breakpoint'}
            } if hasattr(__builtins__, '__dict__') else {},
            # 数学ライブラリを直接使えるようにする
            'sympy': sympy,
            'np': np,
            'math': math,
        }
        # sympyの全シンボルを展開
        exec('from sympy import *', self.global_env)
        exec('from math import *', self.global_env)
        exec('import numpy as np', self.global_env)
    
    def execute(self, code: str) -> tuple[str, bool]:
        """
        コードを実行して (出力文字列, 成功フラグ) を返す
        """
        stdout_capture = StringIO()
        stderr_capture = StringIO()
        
        def run_code():
            with contextlib.redirect_stdout(stdout_capture), \
                 contextlib.redirect_stderr(stderr_capture):
                exec(code, self.global_env.copy())
        
        try:
            with ThreadPoolExecutor(max_workers=1) as executor:
                future = executor.submit(run_code)
                future.result(timeout=self.timeout)
            output = stdout_capture.getvalue()
            if not output:
                output = stderr_capture.getvalue()
            return output.strip(), True
        except FuturesTimeoutError:
            return f'TimeoutError: コードが{self.timeout}秒以内に完了しませんでした', False
        except Exception as e:
            tb = traceback.format_exc()
            # トレースバックをモデルにフィードバックして自己修正を促す
            return f'{type(e).__name__}: {e}\n{tb[-500:]}', False
    
    def extract_and_execute(self, text: str) -> tuple[str, str, bool]:
        """
        テキストからPythonコードブロックを抽出して実行する
        
        Returns:
            (コード, 実行結果, 成功フラグ)
        """
        # ```python ... ``` ブロックを抽出
        pattern = r'```python\s*([\s\S]*?)\s*```'
        matches = re.findall(pattern, text)
        
        if not matches:
            # フォールバック: ``` ... ``` も試す
            pattern = r'```\s*([\s\S]*?)\s*```'
            matches = re.findall(pattern, text)
        
        if not matches:
            return '', '', False
        
        # 最後のコードブロックを使用
        code = matches[-1].strip()
        output, success = self.execute(code)
        return code, output, success


# テスト
executor = SafeCodeExecutor(timeout=config.code_timeout)
test_code = """
from sympy import *
x = symbols('x')
result = solve(x**2 - 5*x + 6, x)
print(f'解: {result}')
print(f'答え: {min(result)}')
"""
output, success = executor.execute(test_code)
print(f'✅ コード実行テスト:')
print(f'  出力: {output}')
print(f'  成功: {success}')

## 🤖 Step 5: プロンプトエンジニアリング

AIMO問題の特性に合わせた最適なプロンプトを設計します。

In [ ]:
class PromptBuilder:
    """
    AIMO3向けプロンプト生成クラス
    
    TIRモード: モデルにPythonコードを書かせて数値計算を補助
    CoTモード: 純粋な推論チェーン
    """
    
    # ==========================================
    # TIR システムプロンプト
    # Project Numina の戦略 + AIMO3固有の指示
    # ==========================================
    TIR_SYSTEM_PROMPT = """\
あなたは国際数学オリンピック(IMO)レベルの問題を解く数学の専門家です。
与えられた問題を解くために、数学的推論とPythonコードを組み合わせて使用してください。

## 解答フォーマット
1. まず問題を分析し、解法の方針を立ててください
2. 計算が必要な場合は、必ずPythonコードブロックで実行してください:
   ```python
   # ここにコードを書く
   print(結果)
   ```
3. コードの出力を確認し、さらに推論を続けてください
4. 最終的な答えは必ず \\boxed{答え} の形式で記述してください

## 重要な注意事項
- 答えは非負整数で、100000 (10^5) 未満です
- 答えが大きい場合は問題文の指示に従って mod を取ってください
- sympyを積極的に活用してください (solve, factor, simplify, etc.)
- 数値計算はコードで確認し、手計算のミスを防いでください
- 問題は英語で書かれていますが、推論は日本語でも英語でも構いません
"""

    # ==========================================
    # TIR ユーザープロンプト
    # ==========================================
    TIR_USER_TEMPLATE = """\
以下の数学オリンピックの問題を解いてください。
数学的推論とPythonコードの両方を活用して、正確な答えを求めてください。

## 問題
{problem}

## 指示
- step-by-step で推論してください
- 計算はPythonコードで確認してください
- 最終答えは \\boxed{{答え}} に記入してください
- 答えは非負整数です
"""

    # ==========================================
    # TIR コード実行後の継続プロンプト
    # ==========================================
    TIR_EXECUTION_TEMPLATE = """\
```output
{output}
```
"""

    # ==========================================
    # CoT プロンプト (フォールバック)
    # ==========================================
    COT_USER_TEMPLATE = """\
Solve the following math olympiad problem step by step.
Please reason carefully and put your final answer within \\boxed{{}}.
The answer is a non-negative integer less than 100000.

Problem: {problem}

Let me think step by step:
"""

    def __init__(self, model_name: str):
        self.model_name = model_name
        # モデルに応じてプロンプト形式を調整
        self.is_qwen3 = 'Qwen3' in model_name
        self.is_deepseek_r1 = 'R1' in model_name or 'r1' in model_name
    
    def build_tir_messages(self, problem: str) -> list[dict]:
        """TIRモードの初期メッセージを構築"""
        messages = []
        
        # DeepSeek-R1はsystem promptなしを推奨
        if not self.is_deepseek_r1:
            messages.append({
                'role': 'system',
                'content': self.TIR_SYSTEM_PROMPT
            })
        
        messages.append({
            'role': 'user',
            'content': self.TIR_USER_TEMPLATE.format(problem=problem)
        })
        return messages
    
    def build_cot_messages(self, problem: str) -> list[dict]:
        """CoTモードのメッセージを構築"""
        messages = []
        if not self.is_deepseek_r1:
            messages.append({
                'role': 'system',
                'content': 'You are an expert mathematician specializing in mathematical olympiad problems. Solve problems step by step.'
            })
        messages.append({
            'role': 'user',
            'content': self.COT_USER_TEMPLATE.format(problem=problem)
        })
        return messages
    
    def add_code_output(self, messages: list[dict], 
                        assistant_response: str, 
                        code_output: str) -> list[dict]:
        """コード実行結果をメッセージに追加"""
        messages = messages.copy()
        messages.append({'role': 'assistant', 'content': assistant_response})
        messages.append({
            'role': 'user',
            'content': self.TIR_EXECUTION_TEMPLATE.format(output=code_output)
        })
        return messages


# テスト
builder = PromptBuilder(config.primary_model)
sample_problem = "Find the number of positive integers n ≤ 1000 such that n^2 + 3n + 2 is divisible by 7."
messages = builder.build_tir_messages(sample_problem)
print('✅ プロンプト構築テスト:')
print(f'  メッセージ数: {len(messages)}')
print(f'  最初のメッセージ役割: {messages[0]["role"]}')
print(f'  ユーザープロンプト (冒頭): {messages[-1]["content"][:100]}...')

## 🔍 Step 6: 答え抽出 (Answer Extraction)

モデルの出力から最終答えを確実に抽出する頑健なパーサーを実装します。

In [ ]:
class AnswerExtractor:
    """
    モデル出力から数値答えを抽出するクラス
    
    優先順位:
    1. \boxed{数値} パターン
    2. 'The answer is X' パターン
    3. 最後の数値
    """
    
    # \boxed{...} パターン (ネストした {} も考慮)
    BOXED_PATTERNS = [
        r'\\boxed\{([^{}]*)\}',
        r'\\boxed\{([^{}]*\{[^{}]*\}[^{}]*)\}',
        r'\$\\boxed\{([^{}]*)\}\$',
        r'\[\\boxed\{([^{}]*)\}\]',
    ]
    
    # 「答えはX」系パターン
    ANSWER_PATTERNS = [
        r'(?:the\s+)?(?:final\s+)?answer\s+is\s+[:\s]*([\-]?\d+(?:[,\s]\d+)*)',
        r'(?:therefore|thus|hence|so)\s*,?\s*(?:the\s+)?(?:answer|result|value)\s+is\s+([\-]?\d+)',
        r'=\s*([\-]?\d+)\s*$',
        r'答え[は:：]\s*([\-]?\d+)',
    ]
    
    def extract(self, text: str, mod: int = 100000) -> Optional[int]:
        """
        テキストから最終答えを抽出し、整数として返す
        
        Args:
            text: モデルの出力テキスト
            mod: AIMO3は100000
        
        Returns:
            整数の答え (失敗時はNone)
        """
        if not text:
            return None
        
        # 1. \boxed{} から抽出 (最も信頼性が高い)
        for pattern in self.BOXED_PATTERNS:
            matches = re.findall(pattern, text, re.IGNORECASE)
            if matches:
                # 最後の\boxedを使用 (最終答えが最後に来ることが多い)
                raw = matches[-1].strip()
                num = self._parse_number(raw)
                if num is not None:
                    return num % mod
        
        # 2. 「答えはX」系パターン
        for pattern in self.ANSWER_PATTERNS:
            matches = re.findall(pattern, text, re.IGNORECASE | re.MULTILINE)
            if matches:
                raw = matches[-1].strip().replace(',', '').replace(' ', '')
                num = self._parse_number(raw)
                if num is not None:
                    return num % mod
        
        # 3. テキスト末尾付近の数値を探す (最終手段)
        last_200 = text[-200:]
        numbers = re.findall(r'(?<![\d.])([\-]?\d{1,10})(?![\d.])', last_200)
        if numbers:
            for num_str in reversed(numbers):
                num = self._parse_number(num_str)
                if num is not None and 0 <= num < mod:
                    return num % mod
        
        return None
    
    def _parse_number(self, text: str) -> Optional[int]:
        """文字列を整数に変換 (分数、小数点も考慮)"""
        text = text.strip().replace(',', '')
        
        # 整数の直接変換
        try:
            return int(text)
        except ValueError:
            pass
        
        # 分数 (a/b)
        try:
            from fractions import Fraction
            f = Fraction(text)
            if f.denominator == 1:
                return int(f.numerator)
        except (ValueError, ZeroDivisionError):
            pass
        
        # 小数点 (整数部分のみ)
        try:
            f = float(text)
            if f == int(f):
                return int(f)
        except ValueError:
            pass
        
        # sympyで評価
        try:
            result = sympify(text)
            if result.is_integer:
                return int(result)
        except Exception:
            pass
        
        return None


# テスト
extractor = AnswerExtractor()
test_cases = [
    (r"The answer is \boxed{12345}", 12345),
    (r"Therefore, the final answer is \boxed{\frac{100}{1}} = 100", 100),
    (r"We find that n = 42. The answer is 42.", 42),
    (r"計算の結果: \boxed{99999}", 99999),
]
print('✅ Answer Extractor テスト:')
for text, expected in test_cases:
    result = extractor.extract(text)
    status = '✅' if result == expected else '❌'
    print(f'  {status} 入力: ...{text[-30:]} → 抽出: {result} (期待: {expected})')

## 🚀 Step 7: vLLM 推論エンジン

vLLMを使って高スループットで推論します。H100 x2 でテンソル並列化することで、32Bモデルを効率的に動かします。

In [ ]:
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

class VLLMInferenceEngine:
    """
    vLLMベースの高速推論エンジン
    
    特徴:
    - テンソル並列化 (H100 x2)
    - プレフィックスキャッシュで同一問題の複数サンプリングを高速化
    - バッチ処理で全問題を効率的に推論
    """
    
    def __init__(self, model_name: str, config: AIMOConfig):
        self.model_name = model_name
        self.config = config
        print(f'🔄 Loading model: {model_name}')
        print(f'   Tensor parallel size: {config.tensor_parallel_size}')
        
        # vLLMエンジンの初期化
        self.llm = LLM(
            model=model_name,
            tensor_parallel_size=config.tensor_parallel_size,
            max_model_len=config.max_model_len,
            gpu_memory_utilization=config.gpu_memory_utilization,
            dtype=config.dtype,
            enable_prefix_caching=config.enable_prefix_caching,
            trust_remote_code=True,
            # Qwen3はthinking modeをサポート
            # 'enable_thinking': True  # Qwen3専用
        )
        self.tokenizer = self.llm.get_tokenizer()
        print(f'✅ Model loaded: {model_name}')
    
    def generate_batch(
        self,
        messages_list: list[list[dict]],
        n: int = 1,
        temperature: float = 0.6,
        max_tokens: int = 8192,
        stop_sequences: list[str] = None,
    ) -> list[list[str]]:
        """
        複数の問題を一括して推論
        
        Args:
            messages_list: 問題ごとのメッセージリスト
            n: 各問題で生成するサンプル数
            temperature: サンプリング温度
            max_tokens: 最大生成トークン数
        
        Returns:
            [[candidates for problem 1], [candidates for problem 2], ...]
        """
        # メッセージをトークン化
        prompts = []
        for messages in messages_list:
            prompt = self.tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )
            prompts.append(prompt)
        
        # サンプリングパラメータの設定
        stop = stop_sequences or ['```output']
        sampling_params = SamplingParams(
            n=n,
            temperature=temperature,
            top_p=self.config.top_p,
            max_tokens=max_tokens,
            stop=stop,
        )
        
        # バッチ推論
        outputs = self.llm.generate(prompts, sampling_params)
        
        # 結果を問題ごとに整理
        results = []
        for output in outputs:
            candidates = [o.text for o in output.outputs]
            results.append(candidates)
        
        return results
    
    def generate_single(
        self,
        messages: list[dict],
        n: int = 1,
        temperature: float = 0.6,
        max_tokens: int = 8192,
        stop_sequences: list[str] = None,
    ) -> list[str]:
        """
        単一問題の推論
        """
        results = self.generate_batch(
            [messages], n=n, temperature=temperature,
            max_tokens=max_tokens, stop_sequences=stop_sequences
        )
        return results[0]


print('✅ VLLMInferenceEngine クラス定義完了')
print('   ※ モデルの実際のロードは次のセルで行います')

In [ ]:
# ==========================================
# モデルのロード
# ==========================================
# Kaggle では /kaggle/input/ にモデルを事前配置しておく
# または Hugging Face Hub から直接ダウンロード

# モデルパスの設定
# Kaggle Dataset として登録済みの場合:
# PRIMARY_MODEL_PATH = '/kaggle/input/qwen3-32b/'
# Hugging Face Hub から:
PRIMARY_MODEL_PATH = config.primary_model  # 'Qwen/Qwen3-32B'
SECONDARY_MODEL_PATH = config.secondary_model  # 'deepseek-ai/DeepSeek-R1-Distill-Qwen-32B'

print(f'Primary model: {PRIMARY_MODEL_PATH}')
print(f'Secondary model: {SECONDARY_MODEL_PATH}')
print('\n⚠️  モデルのロードには数分かかります (H100 x2 で約2-3分)')

# Primaryモデルをロード
primary_engine = VLLMInferenceEngine(PRIMARY_MODEL_PATH, config)

# PromptBuilderとAnswerExtractorのインスタンス化
primary_builder = PromptBuilder(PRIMARY_MODEL_PATH)
extractor = AnswerExtractor()
code_executor = SafeCodeExecutor(timeout=config.code_timeout)

print('\n✅ 全コンポーネント準備完了')

## 🔄 Step 8: TIRパイプライン (コア推論ループ)

Project Numinaが1位を取った手法: **SC-TIR (Self-Consistency with Tool-Integrated Reasoning)**

```
For each problem:
  1. N個の候補を並列生成
  2. 各候補でコードブロックを検出・実行
  3. 実行結果をフィードバックして継続生成 (depth M回)
  4. 最終答えを抽出
  5. 多数決で最終答えを決定
```

In [ ]:
class SCTIRPipeline:
    """
    SC-TIR (Self-Consistency + Tool-Integrated Reasoning) パイプライン
    
    AIMO2 1位のProject Numinaが開発した手法を拡張:
    - N=48 候補を並列生成 (多数決のベース)
    - depth=4 のコード実行ループ (自己修正)
    - 最終答えは加重多数決
    """
    
    def __init__(
        self,
        engine: VLLMInferenceEngine,
        builder: PromptBuilder,
        extractor: AnswerExtractor,
        executor: SafeCodeExecutor,
        config: AIMOConfig,
    ):
        self.engine = engine
        self.builder = builder
        self.extractor = extractor
        self.executor = executor
        self.config = config
    
    def solve_problem(
        self,
        problem: str,
        problem_id: int = 0,
        verbose: bool = False,
    ) -> dict:
        """
        1問を解く
        
        Returns:
            {
                'answer': int,          # 最終答え
                'confidence': float,    # 信頼度 (多数決の割合)
                'all_answers': list,    # 全候補の答え
                'vote_counts': dict,    # 答えごとの票数
            }
        """
        if verbose:
            print(f'\n=== Problem {problem_id} ===')
            print(f'問題: {problem[:100]}...')
        
        N = self.config.num_candidates
        M = self.config.max_tir_depth
        
        # ==========================================
        # Phase 1: 初期候補生成
        # N個のプロンプトを並列生成
        # ==========================================
        initial_messages = self.builder.build_tir_messages(problem)
        
        # N個同時生成 (vLLMのn引数を使う)
        # stop=```output で停止 → コード実行 → 継続
        initial_responses = self.engine.generate_single(
            initial_messages,
            n=N,
            temperature=self.config.temperature,
            max_tokens=self.config.max_tokens,
            stop_sequences=['```output'],
        )
        
        if verbose:
            print(f'  生成した候補数: {len(initial_responses)}')
        
        # ==========================================
        # Phase 2: TIRループ (各候補で depth回繰り返す)
        # ==========================================
        
        # 各候補の状態を追跡
        candidate_states = []
        for resp in initial_responses:
            candidate_states.append({
                'messages': initial_messages.copy(),
                'current_response': resp,
                'full_response': resp,
                'done': False,  # \boxedが見つかったら終了
                'depth': 0,
            })
        
        for depth in range(M):
            # 未完了の候補を抽出
            active_indices = [
                i for i, s in enumerate(candidate_states)
                if not s['done']
            ]
            
            if not active_indices:
                break
            
            # 各候補のコードを実行し、次のメッセージを構築
            next_messages_list = []
            active_states = []
            
            for i in active_indices:
                state = candidate_states[i]
                resp = state['current_response']
                
                # \boxedが見つかったら終了
                if r'\boxed' in resp:
                    state['done'] = True
                    continue
                
                # コードブロックを抽出・実行
                code, output, success = self.executor.extract_and_execute(resp)
                
                if not code:
                    # コードブロックなし → CoT継続を試みる
                    state['done'] = True
                    continue
                
                # 実行結果をメッセージに追加
                new_messages = self.builder.add_code_output(
                    state['messages'], resp, output
                )
                next_messages_list.append(new_messages)
                active_states.append(i)
            
            if not next_messages_list:
                break
            
            # 次のレスポンスを一括生成
            next_responses = self.engine.generate_batch(
                next_messages_list,
                n=1,  # 各候補は1つずつ
                temperature=self.config.temperature,
                max_tokens=self.config.max_tokens,
                stop_sequences=['```output'],
            )
            
            # 状態を更新
            for j, i in enumerate(active_states):
                state = candidate_states[i]
                new_resp = next_responses[j][0]
                state['messages'] = next_messages_list[j]
                state['current_response'] = new_resp
                state['full_response'] += new_resp
                state['depth'] = depth + 1
        
        # ==========================================
        # Phase 3: 答えの抽出と多数決
        # ==========================================
        all_answers = []
        for state in candidate_states:
            answer = self.extractor.extract(
                state['full_response'],
                mod=self.config.answer_mod
            )
            all_answers.append(answer)
        
        # None を除いた有効な答えで多数決
        valid_answers = [a for a in all_answers if a is not None]
        
        if not valid_answers:
            if verbose:
                print('  ⚠️  有効な答えが見つかりませんでした → 0を返す')
            return {
                'answer': 0,
                'confidence': 0.0,
                'all_answers': all_answers,
                'vote_counts': {},
            }
        
        vote_counts = Counter(valid_answers)
        best_answer = vote_counts.most_common(1)[0][0]
        confidence = vote_counts[best_answer] / len(valid_answers)
        
        if verbose:
            print(f'  有効候補数: {len(valid_answers)}/{len(all_answers)}')
            print(f'  上位答え: {vote_counts.most_common(3)}')
            print(f'  最終答え: {best_answer} (confidence: {confidence:.2%})')
        
        return {
            'answer': best_answer,
            'confidence': confidence,
            'all_answers': all_answers,
            'vote_counts': dict(vote_counts),
        }


# パイプラインのインスタンス化
pipeline = SCTIRPipeline(
    engine=primary_engine,
    builder=primary_builder,
    extractor=extractor,
    executor=code_executor,
    config=config,
)
print('✅ SC-TIRパイプライン準備完了')

## 🎯 Step 9: マルチモデルアンサンブル

2つのモデルの予測を組み合わせて精度を上げます。

In [ ]:
class EnsemblePipeline:
    """
    複数モデルの予測をアンサンブルするパイプライン
    
    戦略:
    1. 各モデルがN/2個の候補を生成 (合計N候補)
    2. 各候補から答えを抽出
    3. モデルの重み付き多数決で最終答えを決定
    
    メモリ効率: primaryとsecondaryを順番にロード
    """
    
    def __init__(self, config: AIMOConfig):
        self.config = config
        self.extractor = AnswerExtractor()
        self.executor = SafeCodeExecutor(timeout=config.code_timeout)
    
    def weighted_vote(
        self,
        primary_result: dict,
        secondary_result: dict,
    ) -> dict:
        """
        2つのモデルの結果を重み付き多数決で統合
        
        重みは confidence × model_weight で計算
        """
        # 全候補を収集
        all_valid = []
        
        # Primary モデルの票
        for ans, cnt in primary_result.get('vote_counts', {}).items():
            all_valid.extend([ans] * int(cnt * self.config.primary_weight * 10))
        
        # Secondary モデルの票
        for ans, cnt in secondary_result.get('vote_counts', {}).items():
            all_valid.extend([ans] * int(cnt * self.config.secondary_weight * 10))
        
        if not all_valid:
            # フォールバック: primaryの答えを使用
            return primary_result
        
        vote_counts = Counter(all_valid)
        best_answer = vote_counts.most_common(1)[0][0]
        
        return {
            'answer': best_answer,
            'primary_answer': primary_result.get('answer'),
            'secondary_answer': secondary_result.get('answer'),
            'primary_confidence': primary_result.get('confidence', 0),
            'secondary_confidence': secondary_result.get('confidence', 0),
            'vote_counts': dict(vote_counts),
        }
    
    def solve_with_fallback(
        self,
        primary_pipeline: SCTIRPipeline,
        problem: str,
        problem_id: int = 0,
        low_confidence_threshold: float = 0.3,
        verbose: bool = True,
    ) -> dict:
        """
        Primary モデルで解き、信頼度が低い場合のみ Secondary を使う
        
        (コスト効率を考慮したアダプティブアンサンブル)
        """
        # Primary モデルで解く
        primary_result = primary_pipeline.solve_problem(
            problem, problem_id, verbose=verbose
        )
        
        if verbose:
            print(f'  Primary: {primary_result["answer"]} '
                  f'(confidence: {primary_result["confidence"]:.2%})')
        
        # 信頼度が高ければPrimaryの答えをそのまま使う
        if primary_result['confidence'] >= low_confidence_threshold:
            return primary_result
        
        # 信頼度が低い場合: Secondary モデルを追加で実行
        if verbose:
            print(f'  ⚠️  低信頼度 → Secondary モデルで再確認')
        
        # Secondary エンジンが存在する場合のみ
        if hasattr(self, 'secondary_pipeline'):
            secondary_result = self.secondary_pipeline.solve_problem(
                problem, problem_id, verbose=verbose
            )
            return self.weighted_vote(primary_result, secondary_result)
        
        return primary_result


ensemble = EnsemblePipeline(config)
print('✅ アンサンブルパイプライン準備完了')

## ⚡ Step 10: メイン推論ループ & 提出

全問題を解き、AIMO3 APIに提出します。

In [ ]:
def run_aimo3_pipeline(
    problems_df: pd.DataFrame,
    pipeline: SCTIRPipeline,
    config: AIMOConfig,
    verbose: bool = True,
) -> pd.DataFrame:
    """
    AIMO3 全問題を解くメインループ
    
    Returns:
        提出用DataFrame (id, answer)
    """
    results = []
    total = len(problems_df)
    
    print(f'🚀 AIMO3 推論開始: {total}問')
    print(f'   モデル: {config.primary_model}')
    print(f'   候補数: {config.num_candidates}, 深さ: {config.max_tir_depth}')
    print('=' * 60)
    
    start_time = time.time()
    
    for idx, row in problems_df.iterrows():
        problem_id = row['id']
        problem_text = row['problem']
        
        t0 = time.time()
        print(f'\n[{idx+1}/{total}] Problem ID: {problem_id}')
        print(f'問題 (冒頭): {problem_text[:80]}...')
        
        try:
            result = pipeline.solve_problem(
                problem=problem_text,
                problem_id=problem_id,
                verbose=verbose,
            )
            
            answer = result['answer']
            confidence = result['confidence']
            elapsed = time.time() - t0
            
            print(f'  ✅ 答え: {answer:05d} '
                  f'(信頼度: {confidence:.2%}, 経過: {elapsed:.1f}秒)')
            
            # ローカル評価モードの場合、正解と比較
            if 'answer' in row and not config.is_submission:
                gt = int(row['answer']) % config.answer_mod
                correct = (answer == gt)
                print(f'  正解: {gt:05d} → {"✅" if correct else "❌"}')
            
            results.append({
                'id': problem_id,
                'answer': answer,
                'confidence': confidence,
                'elapsed_sec': elapsed,
            })
            
        except Exception as e:
            print(f'  ❌ エラー: {e}')
            traceback.print_exc()
            results.append({
                'id': problem_id,
                'answer': 0,  # デフォルト答え
                'confidence': 0.0,
                'elapsed_sec': time.time() - t0,
            })
    
    total_time = time.time() - start_time
    results_df = pd.DataFrame(results)
    
    print('\n' + '=' * 60)
    print(f'✅ 推論完了!')
    print(f'   総問題数: {total}')
    print(f'   総時間: {total_time/60:.1f}分')
    print(f'   平均時間/問: {total_time/total:.1f}秒')
    print(f'   平均信頼度: {results_df["confidence"].mean():.2%}')
    
    # ローカル評価モードのスコア計算
    if 'answer' in problems_df.columns and not config.is_submission:
        gt_answers = problems_df['answer'].apply(
            lambda x: int(x) % config.answer_mod
        ).values
        pred_answers = results_df['answer'].values
        accuracy = (gt_answers == pred_answers).mean()
        print(f'   ローカル精度: {accuracy:.2%} ({int(accuracy*total)}/{total})')
    
    return results_df


print('✅ メインループ関数定義完了')

In [ ]:
# ==========================================
# 推論実行
# ==========================================
results_df = run_aimo3_pipeline(
    problems_df=problems_df,
    pipeline=pipeline,
    config=config,
    verbose=True,
)

results_df.head(10)

## 📤 Step 11: AIMO3 API への提出

In [ ]:
# ==========================================
# AIMO3 公式 API への提出
# ==========================================
if config.is_submission:
    import aimo
    
    print('📤 AIMO3 APIへ提出中...')
    
    submission_df = results_df[['id', 'answer']].copy()
    # 答えが5桁の整数であることを確認
    submission_df['answer'] = submission_df['answer'].astype(int).clip(0, 99999)
    
    print('提出データのサンプル:')
    print(submission_df.head())
    
    # APIに提出
    aimo.submit(submission_df)
    print('✅ 提出完了！')

else:
    # ローカル評価: CSVに保存
    output_path = 'aimo3_submission.csv'
    results_df[['id', 'answer']].to_csv(output_path, index=False)
    print(f'✅ 結果を保存: {output_path}')
    
    # 詳細レポート
    print('\n📊 詳細結果:')
    print(results_df.describe())

## 🔬 Step 12: (オプション) Fine-Tuning パイプライン

AIMO3では最大128枚のH100が利用可能 (Fields Model Initiative 経由)。
以下は、数学データでのファインチューニングスクリプトです。

In [ ]:
# ==========================================
# ファインチューニング設定
# (AIMO3 Fields Model Initiative の H100 x128 を使用)
# ==========================================

FINETUNE_CONFIG = """
# 推奨ファインチューニング戦略 (AIMO2上位解法を参考)

## Stage 1: SFT (Supervised Fine-Tuning) on Math CoT
# データセット:
#   - NuminaMath-CoT (AIMO1 1位 Project Numina 公開)
#   - OpenR1-Math-220k (高品質数学CoTデータ)
#   - AIME/AMC/Olympiad 問題集
#   - 自作IMO問題 (GPT-4oでCoT生成)
#
# 設定:
#   - ベースモデル: Qwen3-32B
#   - エポック: 3
#   - 学習率: 1e-5 (cosine decay)
#   - バッチサイズ: 128
#   - max_length: 8192

## Stage 2: TIR Fine-Tuning
# データセット:
#   - NuminaMath-TIR (コード実行ありの推論データ)
#   - ToRAフォーマットで生成した合成データ
#
# 設定:
#   - ベースモデル: Stage1のチェックポイント
#   - エポック: 2
#   - 学習率: 5e-6

## Stage 3: GRPO (強化学習)
# AIMO2の上位チームが採用した手法
# 報酬: 正解 +1, 不正解 -1, コンパイルエラー -0.5
# ベースモデル: Stage2のチェックポイント
"""

print(FINETUNE_CONFIG)

# --- 実際のファインチューニングコード ---
# from trl import SFTTrainer, SFTConfig
# from datasets import load_dataset
# 
# # データ準備
# dataset = load_dataset('AI-MO/NuminaMath-CoT')
# dataset_tir = load_dataset('AI-MO/NuminaMath-TIR')
# 
# # SFT Stage 1
# training_args = SFTConfig(
#     output_dir='./sft_stage1',
#     num_train_epochs=3,
#     per_device_train_batch_size=1,
#     gradient_accumulation_steps=16,
#     learning_rate=1e-5,
#     lr_scheduler_type='cosine',
#     warmup_ratio=0.03,
#     bf16=True,
#     max_seq_length=8192,
#     save_strategy='epoch',
#     logging_steps=10,
# )
# 
# trainer = SFTTrainer(
#     model=model,
#     args=training_args,
#     train_dataset=dataset['train'],
# )
# trainer.train()

## 📊 Step 13: 分析・デバッグツール

In [ ]:
def analyze_results(results_df: pd.DataFrame, problems_df: pd.DataFrame = None):
    """
    推論結果の詳細分析
    """
    print('📊 推論結果分析')
    print('=' * 50)
    
    # 基本統計
    print(f'\n📈 基本統計:')
    print(f'  総問題数: {len(results_df)}')
    print(f'  平均信頼度: {results_df["confidence"].mean():.2%}')
    print(f'  中央値信頼度: {results_df["confidence"].median():.2%}')
    print(f'  平均推論時間: {results_df["elapsed_sec"].mean():.1f}秒/問')
    print(f'  最長推論時間: {results_df["elapsed_sec"].max():.1f}秒')
    
    # 信頼度分布
    print(f'\n📊 信頼度分布:')
    bins = [0, 0.3, 0.5, 0.7, 0.9, 1.01]
    labels = ['<30%', '30-50%', '50-70%', '70-90%', '>90%']
    for i, (lo, hi, label) in enumerate(zip(bins[:-1], bins[1:], labels)):
        count = ((results_df['confidence'] >= lo) & (results_df['confidence'] < hi)).sum()
        bar = '█' * count
        print(f'  {label:8s} | {bar} ({count}問)')
    
    # 低信頼度の問題 (追加調査推奨)
    low_conf = results_df[results_df['confidence'] < 0.3]
    if len(low_conf) > 0:
        print(f'\n⚠️  低信頼度問題 (confidence < 30%): {len(low_conf)}問')
        print(f'   → これらの問題はセカンダリモデルまたは手動確認を推奨')
        print(low_conf[['id', 'answer', 'confidence']].to_string())
    
    # 正解率 (ローカル評価時のみ)
    if problems_df is not None and 'answer' in problems_df.columns:
        merged = results_df.merge(
            problems_df[['id', 'answer']].rename(columns={'answer': 'gt_answer'}),
            on='id'
        )
        merged['gt_answer'] = merged['gt_answer'].astype(int) % config.answer_mod
        merged['correct'] = (merged['answer'] == merged['gt_answer'])
        accuracy = merged['correct'].mean()
        
        print(f'\n🎯 ローカル精度: {accuracy:.2%} ({merged["correct"].sum()}/{len(merged)})')
        
        # 信頼度別の正解率
        print(f'\n🎯 信頼度別正解率:')
        for lo, hi, label in zip(bins[:-1], bins[1:], labels):
            mask = (merged['confidence'] >= lo) & (merged['confidence'] < hi)
            subset = merged[mask]
            if len(subset) > 0:
                acc = subset['correct'].mean()
                print(f'  {label:8s}: {acc:.2%} ({subset["correct"].sum()}/{len(subset)})')


# 結果の分析
analyze_results(results_df, problems_df if not config.is_submission else None)

## 💡 Step 14: 金メダルを狙うための追加戦略

以下は、さらに精度を上げるための高度なテクニックです。

In [ ]:
# ==========================================
# 追加戦略 1: 問題カテゴリ分類 & 特化プロンプト
# ==========================================

CATEGORY_PROMPTS = {
    'algebra': """
This is an ALGEBRA problem. Key techniques to consider:
- Polynomial manipulation, factoring, roots
- AM-GM, Cauchy-Schwarz, other inequalities
- Substitution and transformation
Use sympy.solve(), sympy.factor(), sympy.expand() liberally.
""",
    'number_theory': """
This is a NUMBER THEORY problem. Key techniques to consider:
- Modular arithmetic, Chinese Remainder Theorem
- Prime factorization, GCD/LCM
- Divisibility, Euler's theorem, Fermat's little theorem
Use sympy.factorint(), sympy.ntheory module, and custom mod arithmetic.
""",
    'combinatorics': """
This is a COMBINATORICS problem. Key techniques to consider:
- Counting principles, permutations, combinations
- Generating functions, recurrences
- Inclusion-exclusion, pigeonhole principle
Use sympy.binomial(), itertools, and dynamic programming.
""",
    'geometry': """
This is a GEOMETRY problem. Key techniques to consider:
- Coordinate geometry, trigonometry
- Similar triangles, power of a point
- Complex numbers for geometric transformations
Use sympy.geometry module and coordinate bash with sympy.solve().
""",
}

def classify_problem(problem: str, engine: VLLMInferenceEngine) -> str:
    """
    問題のカテゴリを自動分類する
    """
    classify_messages = [{
        'role': 'user',
        'content': f"""\
Classify this math olympiad problem into one of these categories:
algebra, number_theory, combinatorics, geometry

Problem: {problem}

Answer with ONLY one word (the category name):"""
    }]
    
    response = engine.generate_single(
        classify_messages,
        n=1, temperature=0.1, max_tokens=10
    )[0].strip().lower()
    
    for cat in CATEGORY_PROMPTS.keys():
        if cat in response:
            return cat
    return 'algebra'  # デフォルト


print('✅ カテゴリ特化プロンプト定義完了')

# ==========================================
# 追加戦略 2: 自己検証 (Self-Verification)
# ==========================================

def self_verify_answer(
    problem: str,
    candidate_answer: int,
    engine: VLLMInferenceEngine,
) -> float:
    """
    候補答えの正しさをモデルに検証させる
    
    Returns:
        検証スコア (0.0 - 1.0)
    """
    verify_messages = [{
        'role': 'user',
        'content': f"""\
Verify if the answer to this problem is correct.

Problem: {problem}

Proposed answer: {candidate_answer}

Check the answer step by step using Python if needed.
At the end, respond with exactly one word: CORRECT or WRONG."""
    }]
    
    responses = engine.generate_single(
        verify_messages, n=5, temperature=0.3, max_tokens=2048
    )
    
    correct_count = sum(
        1 for r in responses if 'CORRECT' in r.upper() and 'WRONG' not in r.upper()
    )
    return correct_count / len(responses)

print('✅ 自己検証関数定義完了')

# ==========================================
# 追加戦略 3: 低信頼度問題の再挑戦
# ==========================================

def retry_low_confidence(
    results_df: pd.DataFrame,
    problems_df: pd.DataFrame,
    pipeline: SCTIRPipeline,
    config: AIMOConfig,
    confidence_threshold: float = 0.3,
    extra_candidates: int = 32,
) -> pd.DataFrame:
    """
    信頼度が低い問題に追加サンプルを生成して再推論する
    """
    low_conf_ids = results_df[results_df['confidence'] < confidence_threshold]['id'].tolist()
    
    if not low_conf_ids:
        print('✅ 低信頼度問題なし')
        return results_df
    
    print(f'🔄 {len(low_conf_ids)}問を再推論 (追加サンプル: {extra_candidates})')
    
    # configを一時的に変更
    original_n = config.num_candidates
    config.num_candidates = extra_candidates
    
    retry_df = problems_df[problems_df['id'].isin(low_conf_ids)].copy()
    
    for _, row in retry_df.iterrows():
        result = pipeline.solve_problem(row['problem'], row['id'], verbose=True)
        
        # 元の結果と統合 (元の候補 + 新しい候補で多数決)
        orig = results_df[results_df['id'] == row['id']].iloc[0]
        
        # 統合多数決
        all_answers = result['all_answers']
        valid = [a for a in all_answers if a is not None]
        
        if valid:
            combined_vote = Counter(valid)
            best = combined_vote.most_common(1)[0][0]
            conf = combined_vote[best] / len(valid)
            
            mask = results_df['id'] == row['id']
            results_df.loc[mask, 'answer'] = best
            results_df.loc[mask, 'confidence'] = conf
            print(f'  Problem {row["id"]}: {orig["answer"]} → {best} (confidence: {conf:.2%})')
    
    # configを元に戻す
    config.num_candidates = original_n
    
    return results_df


print('✅ 追加戦略 (再挑戦) 関数定義完了')

In [ ]:
# ==========================================
# 低信頼度問題の再挑戦 (時間に余裕がある場合)
# ==========================================
RETRY_LOW_CONFIDENCE = True  # 再挑戦を行うか

if RETRY_LOW_CONFIDENCE:
    results_df = retry_low_confidence(
        results_df=results_df,
        problems_df=problems_df,
        pipeline=pipeline,
        config=config,
        confidence_threshold=0.35,
        extra_candidates=32,
    )

# 最終分析
print('\n📊 最終結果分析:')
analyze_results(results_df, problems_df if not config.is_submission else None)

In [ ]:
# ==========================================
# 最終提出
# ==========================================
final_submission = results_df[['id', 'answer']].copy()
final_submission['answer'] = final_submission['answer'].astype(int).clip(0, 99999)

print('📋 最終提出データ:')
print(final_submission.to_string())

if config.is_submission:
    import aimo
    aimo.submit(final_submission)
    print('\n🎉 AIMO3 提出完了！金メダル目指して頑張れ！')
else:
    final_submission.to_csv('aimo3_final_submission.csv', index=False)
    print('\n✅ aimo3_final_submission.csv に保存しました')

---

## 📖 まとめ: 金メダル戦略ガイド

### 🏆 コアパイプライン

```
問題入力
   ↓
[カテゴリ分類] → 特化プロンプト選択
   ↓
[vLLM + Qwen3-32B] で N=48 候補を並列生成
   ↓ (各候補ごとに)
[TIRループ x depth=4]
   コード抽出 → Python実行 → 結果フィードバック → 継続生成
   ↓
[答え抽出] \boxed{} から数値を取得
   ↓
[多数決] 最頻出答えを選択
   ↓
[低信頼度チェック] confidence < 30% → 追加32候補で再挑戦
   ↓
最終答え (mod 100000)
```

### 🔑 重要ポイント

| 要素 | 詳細 | 効果 |
|---|---|---|
| **モデル選択** | Qwen3-32B (最強数学推論) | +++++ |
| **TIR** | Python実行で計算検証 | ++++ |
| **N=48多数決** | 確率的ノイズを除去 | +++ |
| **depth=4** | 自己修正で精度向上 | +++ |
| **再挑戦** | 低信頼度問題の追加探索 | ++ |
| **カテゴリ特化** | 問題種別に最適なプロンプト | ++ |
| **ファインチューニング** | 数学データでSFT+GRPO | +++++ |

### 📚 推奨データセット (SFT用)
- `AI-MO/NuminaMath-CoT` - 86万問の数学CoTデータ
- `open-r1/OpenR1-Math-220k` - 高品質IMO/AIME問題
- `HuggingFaceH4/MATH-500` - 検証用

### ⏱️ 時間管理 (AIMO3 H100 x2)
- 1問あたり: 約60-90秒 (N=48, depth=4)
- 55問: 約55-80分 → 制限時間内に十分余裕あり

---
*このノートブックはAIMO2上位解法 (Project Numina, NemoSkills) と最新研究を統合しています。*
*Good luck! 🍀*